# Import Required Libraries
Import pandas for data manipulation, os and glob for file operations, and matplotlib/seaborn for exploratory visualization.

In [3]:
# Import necessary libraries
import pandas as pd  # For data manipulation and analysis
import os  # For file and directory operations
import glob  # For pattern matching in file paths
import matplotlib.pyplot as plt  # For data visualization
import seaborn as sns  # For advanced data visualization

# Configure visualization settings
sns.set(style="whitegrid")  # Set seaborn style for plots
plt.rcParams["figure.figsize"] = (10, 6)  # Set default figure size

# List and Load CSV Files
Use os.listdir() or glob to list all CSV files in the directory, then create a dictionary of dataframes by loading each CSV file with pd.read_csv().

In [4]:
# Define the directory containing the CSV files
data_dir = "/home/understressengineer/programming/SRM_PS1/FEB 2025 Prototype data"

# List all CSV files in the directory using glob
csv_files = glob.glob(os.path.join(data_dir, "*.csv"))

# Create a dictionary to store dataframes for each CSV file
dataframes = {}

# Load each CSV file into a pandas DataFrame and store it in the dictionary
for file_path in csv_files:
    file_name = os.path.basename(file_path)  # Extract the file name
    df_name = os.path.splitext(file_name)[0]  # Remove the file extension
    dataframes[df_name] = pd.read_csv(file_path)  # Load the CSV into a DataFrame

# Display the keys of the dictionary to confirm successful loading
print("Loaded datasets:", list(dataframes.keys()))

/tmp/ipykernel_152509/3899583058.py:14: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframes[df_name] = pd.read_csv(file_path)  # Load the CSV into a DataFrame


Loaded datasets: ['WR', 'NR', 'NCR', 'CR', 'SCR', 'NFR', 'ECOR', 'ECR', 'WCR', 'Test - Zone', 'SR', 'NER', 'SECR']


# Explore Dataset Structure
Examine each dataframe's structure with .info(), .head(), and .describe() to understand column names, data types, and basic statistics. Create a summary of columns across all datasets.

In [5]:
# Explore Dataset Structure

# Iterate through each dataframe in the dictionary
for name, df in dataframes.items():
    print(f"Dataset: {name}")
    print("-" * 40)
    
    # Display the first few rows of the dataframe
    print("First 5 rows:")
    print(df.head())
    print()
    
    # Display the dataframe's structure and data types
    print("DataFrame Info:")
    print(df.info())
    print()
    
    # Display basic statistics for numerical columns
    print("Descriptive Statistics:")
    print(df.describe())
    print()
    
    # Display column names
    print("Columns:")
    print(df.columns.tolist())
    print("=" * 80)

# Create a summary of columns across all datasets
column_summary = {name: df.columns.tolist() for name, df in dataframes.items()}

# Display the summary of columns
print("Summary of Columns Across Datasets:")
for dataset, columns in column_summary.items():
    print(f"{dataset}: {columns}")

Dataset: WR
----------------------------------------
First 5 rows:
                  Time Site Name Point Machine Name Direction  \
0  2025-02-10 01:04:09      Atul            101/102   Reverse   
1  2025-02-10 01:12:10      Atul            101/102    Normal   
2  2025-02-10 07:13:06      Atul            101/102   Reverse   
3  2025-02-10 08:00:53      Atul            101/102   Reverse   
4  2025-02-10 08:03:36      Atul            101/102    Normal   

                                           A Current  \
0  0.0,1.5,4.9,4.5,3.7,3,2.6,2.3,2.1,1.9,1.9,1.8,...   
1  0,5.2,4.9,4,3.3,2.8,2.4,2.2,2,1.9,1.8,1.8,1.8,...   
2  0.0,3.2,4.9,4.2,3.5,2.9,2.5,2.2,2,1.9,2,2,2,2....   
3  0.0,4.5,4.7,3.9,3.2,2.7,2.4,2.1,2,1.9,1.9,1.9,...   
4  0,5.1,5.1,4.1,3.4,2.9,2.5,2.2,2,1.9,1.8,1.8,1....   

                                           A Voltage  \
0  0.0,13.5,46.5,72.0,88.5,99.0,103.5,108.0,109.5...   
1  0.0,88.5,90.0,97.5,102.0,105.0,108.0,109.5,111...   
2  0.0,25.5,55.5,78.0,91.5,100.5,105.

# Check for Common Columns
Identify common columns across datasets that can be used for merging. Analyze column name variations and standardize them if needed.

In [6]:
# Check for Common Columns

# Identify common columns across all datasets
common_columns = set.intersection(*[set(df.columns) for df in dataframes.values()])

# Display the common columns
print("Common Columns Across All Datasets:")
print(common_columns)

# Identify columns that are not common across all datasets
non_common_columns = {name: set(df.columns) - common_columns for name, df in dataframes.items()}

# Display non-common columns for each dataset
print("\nNon-Common Columns in Each Dataset:")
for dataset, columns in non_common_columns.items():
    print(f"{dataset}: {columns}")

# Standardize column names by converting them to lowercase and stripping whitespace
for name, df in dataframes.items():
    df.columns = df.columns.str.lower().str.strip()

# Recheck for common columns after standardization
common_columns_standardized = set.intersection(*[set(df.columns) for df in dataframes.values()])

# Display the updated common columns
print("\nCommon Columns After Standardization:")
print(common_columns_standardized)

Common Columns Across All Datasets:
{'Type of B', 'A Voltage', 'A Current', 'Direction', 'Polling of B', 'Site Name', 'Type of A', 'Polling of A', 'Time', 'Point Machine Name', 'B Voltage', 'B Current'}

Non-Common Columns in Each Dataset:
WR: set()
NR: set()
NCR: set()
CR: set()
SCR: set()
NFR: set()
ECOR: set()
ECR: set()
WCR: set()
Test - Zone: set()
SR: set()
NER: set()
SECR: set()

Common Columns After Standardization:
{'a voltage', 'type of a', 'polling of b', 'point machine name', 'type of b', 'time', 'site name', 'b voltage', 'polling of a', 'b current', 'a current', 'direction'}


# Clean the Datasets
Handle missing values, duplicate records, and inconsistent data types. Standardize column formats and values across all datasets.

In [7]:
# Handle missing values, duplicate records, and inconsistent data types
for name, df in dataframes.items():
    print(f"Cleaning dataset: {name}")
    
    # Handle missing values
    # Drop rows with all NaN values
    df.dropna(how='all', inplace=True)
    
    # Fill missing values with appropriate defaults (e.g., 0 for numerical columns, 'Unknown' for categorical columns)
    for col in df.columns:
        if df[col].dtype == 'object':  # Categorical column
            df[col].fillna('Unknown', inplace=True)
        else:  # Numerical column
            df[col].fillna(0, inplace=True)
    
    # Remove duplicate records
    df.drop_duplicates(inplace=True)
    
    # Ensure consistent data types
    # Convert numerical columns to float
    for col in df.select_dtypes(include=['int64', 'float64']).columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Convert categorical columns to string
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype(str)
    
    # Standardize column formats (e.g., trim whitespace in string columns)
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].str.strip()
    
    print(f"Finished cleaning dataset: {name}\n")

# Verify cleaning results
for name, df in dataframes.items():
    print(f"Post-cleaning summary for dataset: {name}")
    print("-" * 40)
    print("Missing values per column:")
    print(df.isnull().sum())
    print("Number of duplicate rows:", df.duplicated().sum())
    print("=" * 80)

Cleaning dataset: WR


/tmp/ipykernel_152509/954950043.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna('Unknown', inplace=True)
/tmp/ipykernel_152509/954950043.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using

Finished cleaning dataset: WR

Cleaning dataset: NR


/tmp/ipykernel_152509/954950043.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna('Unknown', inplace=True)
/tmp/ipykernel_152509/954950043.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using

Finished cleaning dataset: NR

Cleaning dataset: NCR


...

Finished cleaning dataset: SR

Cleaning dataset: NER


/tmp/ipykernel_152509/954950043.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna('Unknown', inplace=True)
/tmp/ipykernel_152509/954950043.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using

Finished cleaning dataset: NER

Cleaning dataset: SECR


/tmp/ipykernel_152509/954950043.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna('Unknown', inplace=True)
/tmp/ipykernel_152509/954950043.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using

Finished cleaning dataset: SECR

Post-cleaning summary for dataset: WR
----------------------------------------
Missing values per column:
time                  0
site name             0
point machine name    0
direction             0
a current             0
a voltage             0
b current             0
b voltage             0
type of a             0
type of b             0
polling of a          0
polling of b          0
dtype: int64
Number of duplicate rows: 0
Post-cleaning summary for dataset: NR
----------------------------------------
Missing values per column:
time                  0
site name             0
point machine name    0
direction             0
a current             0
a voltage             0
b current             0
b voltage             0
type of a             0
type of b             0
polling of a          0
polling of b          0
dtype: int64
Number of duplicate rows: 0
Post-cleaning summary for dataset: NCR
----------------------------------------
Missing values pe

# Merge the Datasets
Implement appropriate merging strategy based on dataset structure. Use pandas concat() or merge() functions depending on the commonality between datasets.

In [10]:
# Merge the Datasets

# Merge all datasets based on common columns
merged_dataset = pd.concat(
    [df[common_columns_standardized] for df in dataframes.values()],
    axis=0,
    ignore_index=True
)

# Display the shape and first few rows of the merged dataset
print("Merged Dataset Shape:", merged_dataset.shape)
print("First 5 Rows of Merged Dataset:")
print(merged_dataset.head())

# Save the merged dataset to a CSV file for further use
output_file = os.path.join(data_dir, "merged_dataset.csv")
merged_dataset.to_csv(output_file, index=False)
print(f"Merged dataset saved to: {output_file}")

TypeError: Passing a set as an indexer is not supported. Use a list instead.

# Export Combined Dataset
Save the final combined and cleaned dataset to a new CSV file using df.to_csv().

In [9]:
# # Export Combined Dataset

# # Define the output file path for the combined dataset
# output_file_path = os.path.join(data_dir, "final_combined_dataset.csv")

# # Save the cleaned and merged dataset to a CSV file
# merged_dataset.to_csv(output_file_path, index=False)

# # Confirm the export
# print(f"Final combined dataset successfully saved to: {output_file_path}")